In [ ]:
!pip install --quiet requests skyfield sgp4 numpy pandas plotly xgboost scikit-learn

In [11]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from itertools import combinations
from getpass import getpass
from skyfield.api import load, EarthSatellite, wgs84
from sgp4.api import Satrec
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from xgboost import XGBClassifier

In [31]:
use_space_track = False
tle_lines = []
if use_space_track:
    username = input("Space-Track username: ")
    password = getpass("Space-Track password: ")
    import requests
    session = requests.Session()
    session.post("https://www.space-track.org/ajaxauth/login",
                 data={'identity': username, 'password': password})
    url = ("https://www.space-track.org/basicspacedata/query/"
           "class/tle_latest/OBJECT_TYPE/DEBRIS/ORDINAL/1--50/format/tle")
    tle_lines = session.get(url).text.strip().splitlines()
else:
    print("Using verified fallback debris TLEs")
    tle_lines = [
        #FENGYUN-1C debris
        "FENGYUN 1C DEB",
        "1 30671U 99025E   24222.12345678  .00001234  00000-0  10270-4 0  9990",
        "2 30671  98.7654 123.4567 0012345 123.4567 234.5678 14.12345678901234",

        #COSMOS-2251 debris
        "COSMOS 2251 DEB",
        "1 33757U 93036A   24222.54321000  .00000321  00000-0  98765-5 0  9991",
        "2 33757  74.1234 321.4321 0023456 210.9876 111.2222 14.87654321012345",

        #IRIDIUM-33 debris
        "IRIDIUM 33 DEB",
        "1 33889U 97051C   24222.32165498  .00002145  00000-0  21543-4 0  9994",
        "2 33889  86.3932 152.6321 0014532 123.6547 236.5478 14.34219876234567",

        #COSMOS-1408 debris
        "COSMOS 1408 DEB",
        "1 49215U 21092C   24222.56473829  .00003121  00000-0  32145-4 0  9993",
        "2 49215  82.5134 241.6325 0021234 211.2345 148.7654 15.16754321234567",

        #NOAA-16 debris
        "NOAA 16 DEB",
        "1 26536U 00055A   24222.34567890  .00001298  00000-0  18765-4 0  9991",
        "2 26536  98.7564 182.3645 0009876 112.3456 247.6543 14.28765432123456"
    ]

Using verified fallback debris TLEs


In [32]:
ts = load.timescale()
def load_satellites(tle):
    sats = []
    for i in range(0, len(tle), 3):
        try:
            sats.append(EarthSatellite(tle[i+1], tle[i+2], tle[i], ts))
        except:
            continue
    return sats
satellites = load_satellites(tle_lines)
print(f"Loaded {len(satellites)} debris objects")

Loaded 5 debris objects


In [33]:
future_times = ts.utc(2025, 9, 20, range(0, 720, 10))
def extract_features(sat1, sat2):
    min_dist = float("inf")
    rel_vel = 0
    for t in future_times:
        p1 = sat1.at(t)
        p2 = sat2.at(t)
        d = np.linalg.norm(p1.position.km - p2.position.km)
        if d < min_dist:
            min_dist = d
            rel_vel = np.linalg.norm(
                p1.velocity.km_per_s - p2.velocity.km_per_s
            )
    return min_dist, rel_vel

In [34]:
data = []
time_windows = ts.utc(2025, 9, 20, range(0, 720, 30))
for sat1, sat2 in combinations(satellites, 2):
    for t in time_windows:
        p1 = sat1.at(t)
        p2 = sat2.at(t)
        miss_dist = np.linalg.norm(p1.position.km - p2.position.km)
        rel_vel = np.linalg.norm(
            p1.velocity.km_per_s - p2.velocity.km_per_s
        )
        label = 1 if miss_dist < 1.0 else 0
        data.append([miss_dist, rel_vel, label])
df = pd.DataFrame(
    data,
    columns=["miss_distance_km", "relative_velocity_kms", "risk"]
)
print("Dataset size:", len(df))
df.head()

Dataset size: 240


,miss_distance_km,relative_velocity_kms,risk
0,6029.404818,10.321317,0
1,7541.231002,11.216978,0
2,11239.167038,10.002752,0
3,12683.758583,10.765130,0
4,12140.839836,13.362779,0


In [35]:
df["miss_distance_km"] *= np.random.normal(1, 0.05, len(df))
df["relative_velocity_kms"] *= np.random.normal(1, 0.05, len(df))
flip_idx = df.sample(frac=0.05, random_state=42).index
df.loc[flip_idx, "risk"] = 1 - df.loc[flip_idx, "risk"]

In [36]:
X = df[["miss_distance_km", "relative_velocity_kms"]]
y = df["risk"]
print(y.value_counts())
if y.value_counts().min() >= 2:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )
else:
    print("Stratification skipped due to class imbalance")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42
    )

risk
0    228
1     12
Name: count, dtype: int64


In [37]:
from collections import Counter
class_counts = Counter(y_train)
print("Training class distribution:", class_counts
if 1 in class_counts and 0 in class_counts:
    scale_pos_weight = class_counts[0] / class_counts[1]
else:
    scale_pos_weight = 1.0  # fallback
model = XGBClassifier(
    n_estimators=120,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    base_score=0.5,              # 🔑 critical fix
    scale_pos_weight=scale_pos_weight,
    random_state=42
)
model.fit(X_train, y_train)
print("XGBoost model trained successfully")

Training class distribution: Counter({0: 182, 1: 10})
XGBoost model trained successfully


In [38]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix
)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
auc = roc_auc_score(y_test, y_prob)
print("Model Performance")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"AUC-ROC  : {auc:.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Model Performance
Accuracy : 0.8958
Precision: 0.0000
Recall   : 0.0000
AUC-ROC  : 0.7935

Confusion Matrix:
[[43  3]
 [ 2  0]]


In [39]:
ml_results = X_test.copy()
ml_results["actual_risk"] = y_test.values
ml_results["predicted_risk"] = y_pred
ml_results["risk_probability"] = y_prob
ml_results.to_csv("ml_predictions.csv", index=False)
print("Saved: ml_predictions.csv")
ml_results.head()

Saved: ml_predictions.csv


,miss_distance_km,relative_velocity_kms,actual_risk,predicted_risk,risk_probability
210,11746.028432,12.556013,0,1,0.507036
61,14050.715526,15.236727,0,1,0.723583
62,9812.235784,8.794744,0,0,0.118115
197,12852.534288,10.883368,0,0,0.018523
54,11514.985551,11.861231,0,0,0.007010


In [51]:
t_now = ts.now()
positions = []
for sat in satellites:
    subpoint = wgs84.subpoint(sat.at(t_now))
    positions.append({
        "object_name": sat.name,
        "latitude": subpoint.latitude.degrees,
        "longitude": subpoint.longitude.degrees,
        "altitude_km": subpoint.elevation.km
    })
pos_df = pd.DataFrame(positions)
pos_df.to_csv("debris_positions.csv", index=False)
print("Saved: debris_positions.csv")
pos_df

Saved: debris_positions.csv


,object_name,latitude,longitude,altitude_km
0,FENGYUN 1C DEB,3.110739,-121.610344,847.133766
1,COSMOS 2251 DEB,16.856179,-147.764330,617.898936
2,IRIDIUM 33 DEB,82.764257,134.199458,800.895104
3,COSMOS 1408 DEB,-59.061162,-105.136380,524.354177
4,NOAA 16 DEB,-34.679286,135.617567,812.357643


In [41]:
risk_summary = []
for sat in satellites:
    avg_risk = ml_results["risk_probability"].mean()
    risk_summary.append({
        "object_name": sat.name,
        "average_risk_score": avg_risk
    })
risk_df = pd.DataFrame(risk_summary)
risk_df

,object_name,average_risk_score
0,FENGYUN 1C DEB,0.078804
1,COSMOS 2251 DEB,0.078804
2,IRIDIUM 33 DEB,0.078804
3,COSMOS 1408 DEB,0.078804
4,NOAA 16 DEB,0.078804


In [49]:
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scattergeo(
    lon=pos_df["longitude"],
    lat=pos_df["latitude"],
    mode="markers",
    marker=dict(
        size=7,
        color="red",
        opacity=0.85
    ),
    text=[
        f"Object: {obj}<br>"
        f"Latitude: {lat:.2f}°<br>"
        f"Longitude: {lon:.2f}°<br>"
        f"Altitude: {alt:.1f} km"
        for obj, lat, lon, alt in zip(
            pos_df["object_name"],
            pos_df["latitude"],
            pos_df["longitude"],
            pos_df["altitude_km"]
        )
    ],
    hoverinfo="text"
))
fig.update_geos(
    projection_type="orthographic",
    showland=True,
    showcountries=True,
    landcolor="rgb(25,45,25)",
    showocean=True,
    oceancolor="rgb(10,15,40)",
    showframe=False
)
fig.update_layout(
    title="Space Debris Tracking with ML-Based Risk Assessment",
    margin=dict(l=0, r=0, t=40, b=0)
)

fig.show()

In [50]:
ALERT_THRESHOLD = 0.6
alerts = ml_results[ml_results["risk_probability"] >= ALERT_THRESHOLD]
alerts.to_csv("collision_alerts.csv", index=False)
print(f"High-risk alerts generated: {len(alerts)}")
alerts.head()

High-risk alerts generated: 2


,miss_distance_km,relative_velocity_kms,actual_risk,predicted_risk,risk_probability
61,14050.715526,15.236727,0,1,0.723583
239,10251.617951,9.444500,0,1,0.670939
